# thesis-cpd-final — PROVENANCE & SCOPE (read first)

**Authoritative outputs of this notebook** (both weight-INDEPENDENT):
- cell `PER-NODE RECON ERROR DUMP` → `pernode/{subj}_{split}_pernode.npy` → repo `data/pernode/` (attribution §7 / v3 source)
- cell `SECTION 5B-bis — EXPORT PER-COMPONENT Z-SCORES` → `components/{zrecon,ztemp,zgamma}_{subj}_{split}.npy` → repo `data/processed/components/`

These components reproduce the LOCKED results **bit-exact** when recombined downstream — verified 2026-07-25 (`thesis_repro_lock.py`).

**SUPERSEDED here (do NOT use):** Sections **5B (ensemble)**, **5C (AUROC)**, **5D (ens cache export)** use the
PRE-Decision-#19 ensemble weight (0.35/0.30/0.35) and feed the old `cpd_pipeline_v13`. Per Decision #21 the
ensemble weight lives ONLY in `ensemble_recipe.py` (0.40/0.35/0.25) and the ensemble is built FRESH from the
components above; detection is `cpd_pipeline_v14.py`. The old-weight ens cache these sections produce is
quarantined in the repo and must never be read. Kept below only as a development record. See `docs/PROVENANCE_MAP.md`.

SECTION 0 — INSTALL & CONFIG

In [ ]:
# =============================================================================
# SECTION 0A — INSTALL
# =============================================================================
import subprocess
subprocess.run(["pip", "install", "torch-geometric", "-q"], check=True)

In [ ]:
# =============================================================================
# SECTION 0B — CONFIG (tất cả constants trong 1 cell duy nhất)
# =============================================================================
import os, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data, Batch as PyGBatch
from torch_geometric.utils import dense_to_sparse
from sklearn.metrics import roc_auc_score
import pandas as pd
 
# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
 
# --- Paths (Kaggle dataset mount points) ---
ADJ_DIR     = Path("/kaggle/input/datasets/nhn2mm/chbmit-topk20")
FEAT_DIR    = Path("/kaggle/input/datasets/nhn2mm/chbmit-processed")
MODEL_PATH  = Path("/kaggle/input/datasets/nhn2mm/gae-joint-model/best_model_joint_lambda01.pt")
TEMP_DIR    = Path("/kaggle/input/datasets/nhn2mm/temporal-zscores")
GAMMA_DIR   = Path("/kaggle/input/datasets/nhn2mm/gamma-aec-scores")
SUMMARY_DIR = Path("/kaggle/input/datasets/nhn2mm/chb-mit-summary/summary")
OUT_DIR     = Path("/kaggle/working")
 
# --- Fixed split (seed=42, PERMANENT) ---
TRAIN_SUBJS = ["chb01","chb02","chb04","chb05","chb07","chb08",
               "chb09","chb12","chb19","chb20","chb21","chb23"]
VAL_SUBJS   = ["chb10","chb11","chb22"]
TEST_SUBJS  = ["chb03","chb06","chb13","chb14","chb15","chb16","chb17","chb18"]
 
# --- Model architecture (LOCKED — must match best_model_joint_lambda01.pt) ---
INPUT_DIM  = 23    # A_row_norm(18) + band_powers_norm(5)
HIDDEN_DIM = 64
LATENT_DIM = 16
N_CH       = 18
N_BANDS    = 5
LAMBDA     = 0.1   # joint reconstruction weight: score = MSE(A) + 0.1*MSE(X)
 
# --- Graph config ---
ADJS_SUFFIX = "_topk20"   # files named {subj}_interictal_adjs_topk20.npy
 
# --- Ensemble weights (locked from fine sweep, AUROC 0.7957) ---
W_R = 0.35   # reconstruction (GAE)
W_T = 0.30   # temporal (LSTM)
W_G = 0.35   # gamma AEC
 
# --- Scoring config ---
BATCH_SIZE = 512   # increase if GPU memory allows; 256 is safe minimum
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"MODEL_PATH exists:  {MODEL_PATH.exists()}")
print(f"ADJ_DIR exists:     {ADJ_DIR.exists()}")
print(f"FEAT_DIR exists:    {FEAT_DIR.exists()}")
print(f"TEMP_DIR exists:    {TEMP_DIR.exists()}")
print(f"GAMMA_DIR exists:   {GAMMA_DIR.exists()}")
print(f"SUMMARY_DIR exists: {SUMMARY_DIR.exists()}")

SECTION 1 — MODEL DEFINITION

In [ ]:
# =============================================================================
# SECTION 1A — MODEL DEFINITION
# Architecture must match exactly what was used to train best_model_joint_lambda01.pt
# =============================================================================
class GAEEncoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                 latent_dim=LATENT_DIM):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, latent_dim)
        self.relu  = nn.ReLU()
 
    def forward(self, x, edge_index, edge_weight=None):
        h = self.relu(self.conv1(x, edge_index, edge_weight))
        return self.conv2(h, edge_index, edge_weight)
 
 
class XDecoder(nn.Module):
    """MLP decoder: Z [N_CH, LATENT_DIM] → X_hat [N_CH, N_BANDS]"""
    def __init__(self, latent_dim=LATENT_DIM, n_bands=N_BANDS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_bands)
        )
    def forward(self, z):
        return self.net(z)
 
 
class GAEModel(nn.Module):
    """
    Joint Graph Autoencoder.
    Encoder:   GCNConv(23→64)+ReLU → GCNConv(64→16) → Z ∈ R^[18×16]
    A Decoder: clamp(Z @ Z^T, 0, 1)
    X Decoder: MLP(16→32→5)
    Score:     MSE(A, A_hat) + 0.1 × MSE(X_norm, X_hat)
    """
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                 latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder   = GAEEncoder(input_dim, hidden_dim, latent_dim)
        self.x_decoder = XDecoder(latent_dim, N_BANDS)
 
    def forward(self, x, edge_index, edge_weight=None):
        z    = self.encoder(x, edge_index, edge_weight)
        z_b  = z.unsqueeze(0)
        A_hat = torch.clamp(
            torch.bmm(z_b, z_b.transpose(1, 2)), 0., 1.
        ).squeeze(0)
        X_hat = self.x_decoder(z)
        return z, A_hat, X_hat
 
n_params = sum(p.numel() for p in GAEModel().parameters())
print(f"GAEModel defined. Parameters: {n_params:,}")
print(f"  Architecture: GCNConv({INPUT_DIM}→{HIDDEN_DIM}→{LATENT_DIM})")
print(f"  Score: MSE(A, A_hat) + {LAMBDA}·MSE(X_norm, X_hat)")

In [ ]:
# =============================================================================
# SECTION 1B — LOAD MODEL + VERIFICATION
# =============================================================================
model = GAEModel().to(device)
state = torch.load(str(MODEL_PATH), map_location=device)
model.load_state_dict(state)
model.eval()
 
# Hard verification: random-init model has |bias| < 0.005
_b = model.encoder.conv1.bias
if _b is None or _b.abs().max().item() < 0.005:
    raise RuntimeError(
        f"Bias max={_b.abs().max().item() if _b is not None else 'None'} "
        f"— random init detected. Check MODEL_PATH."
    )
print(f"Model loaded. Bias max = {_b.abs().max().item():.4f}  (expect ~0.8676)")
 
# Sanity check: chb13 AUROC must be ~0.836
def _quick_score(m, adj_path, feat_path):
    """Score a file pair and return raw MSE+lambda*X scores."""
    adjs  = np.load(adj_path,  mmap_mode='r')
    feats = np.load(feat_path, mmap_mode='r')
    scores = []
    m.eval()
    with torch.no_grad():
        for s in range(0, len(adjs), BATCH_SIZE):
            e  = min(s + BATCH_SIZE, len(adjs))
            B  = e - s
            A  = torch.tensor(adjs[s:e].astype(np.float32),  device=device)
            Xt = torch.tensor(feats[s:e].astype(np.float32), device=device)
            An  = A / (A.amax(dim=(1,2), keepdim=True) + 1e-8)
            Xmn = Xt.amin(dim=1, keepdim=True)
            Xmx = Xt.amax(dim=1, keepdim=True)
            Xn  = (Xt - Xmn) / (Xmx - Xmn + 1e-8)
            dl  = [Data(x=torch.cat([An[b], Xn[b]], dim=1),
                        edge_index=dense_to_sparse(A[b])[0],
                        edge_attr=dense_to_sparse(A[b])[1]) for b in range(B)]
            pg  = PyGBatch.from_data_list(dl).to(device)
            z   = model.encoder(pg.x, pg.edge_index, pg.edge_attr)
            zpg = z.view(B, N_CH, LATENT_DIM)
            Ah  = torch.clamp(torch.bmm(zpg, zpg.transpose(1,2)), 0., 1.)
            Xh  = model.x_decoder(z).view(B, N_CH, N_BANDS)
            sc  = ((A - Ah)**2).mean(dim=(1,2)) + \
                  LAMBDA * ((Xn - Xh)**2).mean(dim=(1,2))
            scores.extend(sc.cpu().numpy().tolist())
    return np.array(scores, dtype=np.float32)
 
_si = _quick_score(model,
    str(ADJ_DIR  / f"chb13_interictal_adjs{ADJS_SUFFIX}.npy"),
    str(FEAT_DIR / "chb13_interictal_features.npy"))
_sc = _quick_score(model,
    str(ADJ_DIR  / f"chb13_ictal_adjs{ADJS_SUFFIX}.npy"),
    str(FEAT_DIR / "chb13_ictal_features.npy"))
_y   = np.concatenate([np.zeros(len(_si)), np.ones(len(_sc))])
_auc = roc_auc_score(_y, np.concatenate([_si, _sc]))
print(f"Sanity check chb13 AUROC: {_auc:.4f}  (expected ~0.836)")
if _auc < 0.70:
    raise RuntimeError(
        f"chb13 AUROC={_auc:.4f} too low — wrong model or adjacency files."
    )
print("Sanity check PASSED.\n")
del _si, _sc, _y, _auc  # free memory

SECTION 2 — GAE SCORING

In [ ]:
# =============================================================================
# SECTION 2 — GAE RECONSTRUCTION SCORING
# Score all 8 test subjects using the joint model.
# Output: raw MSE scores (NOT yet z-normalized).
# Z-normalization is done in Section 5 jointly with the other signals.
# =============================================================================
def score_adj_files(adj_path, feat_path):
    """
    Returns raw joint MSE scores per window.
    score = MSE(A, A_hat) + LAMBDA * MSE(X_norm, X_hat)
    No z-normalization here — applied in Section 5.
    """
    adjs  = np.load(adj_path,  mmap_mode='r')
    feats = np.load(feat_path, mmap_mode='r')
    scores = []
    model.eval()
    with torch.no_grad():
        for s in range(0, len(adjs), BATCH_SIZE):
            e  = min(s + BATCH_SIZE, len(adjs))
            B  = e - s
            A  = torch.tensor(adjs[s:e].astype(np.float32),  device=device)
            Xt = torch.tensor(feats[s:e].astype(np.float32), device=device)
            An  = A / (A.amax(dim=(1,2), keepdim=True) + 1e-8)
            Xmn = Xt.amin(dim=1, keepdim=True)
            Xmx = Xt.amax(dim=1, keepdim=True)
            Xn  = (Xt - Xmn) / (Xmx - Xmn + 1e-8)
            dl  = [Data(x=torch.cat([An[b], Xn[b]], dim=1),
                        edge_index=dense_to_sparse(A[b])[0],
                        edge_attr=dense_to_sparse(A[b])[1]) for b in range(B)]
            pg  = PyGBatch.from_data_list(dl).to(device)
            z   = model.encoder(pg.x, pg.edge_index, pg.edge_attr)
            zpg = z.view(B, N_CH, LATENT_DIM)
            Ah  = torch.clamp(torch.bmm(zpg, zpg.transpose(1,2)), 0., 1.)
            Xh  = model.x_decoder(z).view(B, N_CH, N_BANDS)
            sc  = ((A - Ah)**2).mean(dim=(1,2)) + \
                  LAMBDA * ((Xn - Xh)**2).mean(dim=(1,2))
            scores.extend(sc.cpu().numpy().tolist())
    return np.array(scores, dtype=np.float32)
 
 
print("=" * 60)
print("SECTION 2: GAE Reconstruction Scoring")
print("=" * 60)
 
raw_recon_inter = {}
raw_recon_ictal = {}
 
for subj in TEST_SUBJS:
    ai = str(ADJ_DIR  / f"{subj}_interictal_adjs{ADJS_SUFFIX}.npy")
    ac = str(ADJ_DIR  / f"{subj}_ictal_adjs{ADJS_SUFFIX}.npy")
    fi = str(FEAT_DIR / f"{subj}_interictal_features.npy")
    fc = str(FEAT_DIR / f"{subj}_ictal_features.npy")
 
    s_inter = score_adj_files(ai, fi)
    s_ictal = score_adj_files(ac, fc)
 
    raw_recon_inter[subj] = s_inter
    raw_recon_ictal[subj] = s_ictal
 
    print(f"  {subj}: inter={len(s_inter)}, ictal={len(s_ictal)}, "
          f"mean_inter={s_inter.mean():.5f}, mean_ictal={s_ictal.mean():.5f}")
 
print("\nSection 2 complete. Raw reconstruction scores stored in memory.")

In [ ]:
# =============================================================================
# PER-NODE RECON ERROR DUMP (for Phase B attribution) — insert AFTER cell 5
# Mirrors _quick_score EXACTLY, but keeps error per node instead of averaging.
# =============================================================================
pernode_dir = OUT_DIR / "pernode"; pernode_dir.mkdir(parents=True, exist_ok=True)

def _pernode_scores(m, adj_path, feat_path):
    adjs  = np.load(adj_path,  mmap_mode='r')
    feats = np.load(feat_path, mmap_mode='r')
    out = []
    m.eval()
    with torch.no_grad():
        for s in range(0, len(adjs), BATCH_SIZE):
            e = min(s + BATCH_SIZE, len(adjs)); B = e - s
            A  = torch.tensor(adjs[s:e].astype(np.float32),  device=device)
            Xt = torch.tensor(feats[s:e].astype(np.float32), device=device)
            An  = A / (A.amax(dim=(1,2), keepdim=True) + 1e-8)
            Xmn = Xt.amin(dim=1, keepdim=True); Xmx = Xt.amax(dim=1, keepdim=True)
            Xn  = (Xt - Xmn) / (Xmx - Xmn + 1e-8)
            dl  = [Data(x=torch.cat([An[b], Xn[b]], dim=1),
                        edge_index=dense_to_sparse(A[b])[0],
                        edge_attr=dense_to_sparse(A[b])[1]) for b in range(B)]
            pg  = PyGBatch.from_data_list(dl).to(device)
            z   = model.encoder(pg.x, pg.edge_index, pg.edge_attr)
            zpg = z.view(B, N_CH, LATENT_DIM)
            Ah  = torch.clamp(torch.bmm(zpg, zpg.transpose(1,2)), 0., 1.)
            Xh  = model.x_decoder(z).view(B, N_CH, N_BANDS)
            per = ((A - Ah)**2).mean(dim=2) + LAMBDA * ((Xn - Xh)**2).mean(dim=2)  # [B,18]
            out.append(per.cpu().numpy().astype(np.float32))
    return np.concatenate(out, axis=0)   # [n_win, 18]

for subj in TEST_SUBJS:
    for split in ["interictal", "ictal"]:
        pn = _pernode_scores(model,
            str(ADJ_DIR  / f"{subj}_{split}_adjs{ADJS_SUFFIX}.npy"),
            str(FEAT_DIR / f"{subj}_{split}_features.npy"))
        np.save(pernode_dir / f"{subj}_{split}_pernode.npy", pn)
    # self-check: mean over nodes must reproduce the locked scalar recon score
    chk = _pernode_scores(model,
        str(ADJ_DIR  / f"{subj}_interictal_adjs{ADJS_SUFFIX}.npy"),
        str(FEAT_DIR / f"{subj}_interictal_features.npy")).mean(axis=1)
    ref = _quick_score(model,
        str(ADJ_DIR  / f"{subj}_interictal_adjs{ADJS_SUFFIX}.npy"),
        str(FEAT_DIR / f"{subj}_interictal_features.npy"))
    err = float(np.max(np.abs(chk - ref)))
    print(f"{subj}: pernode saved; self-check max|mean_node - scalar| = {err:.2e}")
print("DONE ->", pernode_dir)

SECTION 3 — TEMPORAL LSTM SCORING

In [ ]:
# =============================================================================
# SECTION 3 — TEMPORAL LSTM SCORES
# Load pre-computed raw temporal scores and verify window counts match recon.
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 3: Temporal LSTM Scores")
print("=" * 60)
 
raw_temp_inter = {}
raw_temp_ictal = {}
 
for subj in TEST_SUBJS:
    t_i = np.load(str(TEMP_DIR / f"temporal_{subj}_zinter.npy"))
    t_c = np.load(str(TEMP_DIR / f"temporal_{subj}_zictal.npy"))
 
    raw_temp_inter[subj] = t_i.astype(np.float32)
    raw_temp_ictal[subj] = t_c.astype(np.float32)
 
    # Verify window count alignment
    n_inter_match = len(t_i) == len(raw_recon_inter[subj])
    n_ictal_match = len(t_c) == len(raw_recon_ictal[subj])
    status = "OK" if (n_inter_match and n_ictal_match) else "MISMATCH"
    print(f"  {subj}: inter={len(t_i)}, ictal={len(t_c)}  [{status}]")
    if status == "MISMATCH":
        print(f"    WARNING: recon_inter={len(raw_recon_inter[subj])}, "
              f"recon_ictal={len(raw_recon_ictal[subj])}")
 
print("\nSection 3 complete.")

SECTION 4 — GAMMA AEC SCORING

In [ ]:
# =============================================================================
# SECTION 4 — GAMMA AEC SCORES
# Load pre-computed raw gamma AEC scores and verify window counts.
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 4: Gamma AEC Scores")
print("=" * 60)
 
raw_gamma_inter = {}
raw_gamma_ictal = {}
 
for subj in TEST_SUBJS:
    g_i = np.load(str(GAMMA_DIR / f"gamma_aec_{subj}_inter.npy"))
    g_c = np.load(str(GAMMA_DIR / f"gamma_aec_{subj}_ictal.npy"))
 
    raw_gamma_inter[subj] = g_i.astype(np.float32)
    raw_gamma_ictal[subj] = g_c.astype(np.float32)
 
    n_inter_match = len(g_i) == len(raw_recon_inter[subj])
    n_ictal_match = len(g_c) == len(raw_recon_ictal[subj])
    status = "OK" if (n_inter_match and n_ictal_match) else "MISMATCH"
    print(f"  {subj}: inter={len(g_i)}, ictal={len(g_c)}  [{status}]")
 
print("\nSection 4 complete.")

SECTION 5 — ENSEMBLE & EXPORT

In [ ]:
# =============================================================================
# SECTION 5A — ALL-WINDOW Z-NORMALIZATION
# Apply robust z-norm (median/MAD) to each signal independently,
# using ALL windows (inter + ictal pooled). No ictal labels needed.
# Rationale: ictal ≈ 0.18% of windows → statistics ≈ interictal-only.
# This is the methodologically clean formulation.
# =============================================================================
def robust_z_norm(raw_inter, raw_ictal):
    """
    Z-normalize using median and MAD computed from ALL windows pooled.
    Returns (z_inter, z_ictal) — same length as inputs.
    """
    all_s = np.concatenate([raw_inter, raw_ictal])
    med   = np.median(all_s)
    mad   = np.median(np.abs(all_s - med)) + 1e-9
    return (raw_inter - med) / mad, (raw_ictal - med) / mad
 
 
print("\n" + "=" * 60)
print("SECTION 5A: All-window Z-normalization (all 3 signals)")
print("=" * 60)
 
z_recon_inter = {}; z_recon_ictal = {}
z_temp_inter  = {}; z_temp_ictal  = {}
z_gamma_inter = {}; z_gamma_ictal = {}
 
for subj in TEST_SUBJS:
    # Normalize each signal independently
    z_recon_inter[subj], z_recon_ictal[subj] = robust_z_norm(
        raw_recon_inter[subj], raw_recon_ictal[subj])
 
    z_temp_inter[subj], z_temp_ictal[subj] = robust_z_norm(
        raw_temp_inter[subj], raw_temp_ictal[subj])
 
    z_gamma_inter[subj], z_gamma_ictal[subj] = robust_z_norm(
        raw_gamma_inter[subj], raw_gamma_ictal[subj])
 
    print(f"  {subj}: z-norm applied to all 3 signals")
 
print("\nSection 5A complete.")

### ⚠ SUPERSEDED — Section 5B (ensemble, OLD weight 0.35/0.30/0.35)
Retained for record only. Authoritative ensemble is built downstream from the exported components via
`ensemble_recipe.build_ensemble` (weight 0.40/0.35/0.25, Decision #19). Do not use `ens_inter`/`ens_ictal` from here.
**The next cell (5B-bis component export) IS authoritative.**

In [ ]:
# =============================================================================
# SECTION 5B — 3-WAY ENSEMBLE
# z_ensemble = W_R * z_recon + W_T * z_temp + W_G * z_gamma
# Weights locked from fine sweep: 0.35 / 0.30 / 0.35
# =============================================================================
print("\n" + "=" * 60)
print(f"SECTION 5B: 3-way ensemble  (W_R={W_R}, W_T={W_T}, W_G={W_G})")
print("=" * 60)
 
ens_inter = {}
ens_ictal = {}
 
for subj in TEST_SUBJS:
    # Use minimum window count across all 3 signals (guard against minor misalign)
    ni = min(len(z_recon_inter[subj]),
             len(z_temp_inter[subj]),
             len(z_gamma_inter[subj]))
    nc = min(len(z_recon_ictal[subj]),
             len(z_temp_ictal[subj]),
             len(z_gamma_ictal[subj]))
 
    ens_inter[subj] = (W_R * z_recon_inter[subj][:ni] +
                       W_T * z_temp_inter[subj][:ni]  +
                       W_G * z_gamma_inter[subj][:ni])
 
    ens_ictal[subj] = (W_R * z_recon_ictal[subj][:nc] +
                       W_T * z_temp_ictal[subj][:nc]  +
                       W_G * z_gamma_ictal[subj][:nc])
 
    print(f"  {subj}: ens_inter={len(ens_inter[subj])}, "
          f"ens_ictal={len(ens_ictal[subj])}, "
          f"mzi={np.median(ens_ictal[subj]):.3f}")
 
print("\nSection 5B complete.")

In [ ]:
# SECTION 5B-bis — EXPORT PER-COMPONENT Z-SCORES (for event-tier ablation + B2)
# Saves the EXACT z-arrays used inside the ensemble (no recompute).
from pathlib import Path
comp_dir = OUT_DIR / "components"; comp_dir.mkdir(parents=True, exist_ok=True)
for subj in TEST_SUBJS:
    np.save(comp_dir / f"zrecon_{subj}_inter.npy", z_recon_inter[subj])
    np.save(comp_dir / f"zrecon_{subj}_ictal.npy", z_recon_ictal[subj])
    np.save(comp_dir / f"ztemp_{subj}_inter.npy",  z_temp_inter[subj])
    np.save(comp_dir / f"ztemp_{subj}_ictal.npy",  z_temp_ictal[subj])
    np.save(comp_dir / f"zgamma_{subj}_inter.npy", z_gamma_inter[subj])
    np.save(comp_dir / f"zgamma_{subj}_ictal.npy", z_gamma_ictal[subj])
print("exported per-component z-scores ->", comp_dir)

### ⚠ SUPERSEDED — Section 5C (AUROC on old-weight ensemble)
Expects macro AUROC ~0.7957 (old weight). Current locked macro AUROC is **0.791** (new weight; `window_tier_newweight.py`).
The recon / temporal / gamma STANDALONE columns are weight-independent and remain valid (ROR §2).

In [ ]:
# =============================================================================
# SECTION 5C — AUROC VERIFICATION
# Compute AUROC per subject on ensemble scores.
# Expected: macro AUROC ≈ 0.7957
# If AUROC deviates by > 0.01, something is wrong — do not proceed.
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 5C: AUROC Verification")
print("=" * 60)
 
auroc_rows = []
per_subject_aurocs = []
 
print(f"{'Subj':<6} {'AUROC_ens':>10} {'mzi_ens':>9} "
      f"{'AUROC_recon':>12} {'AUROC_temp':>11} {'AUROC_gamma':>12}")
print("-" * 65)
 
for subj in TEST_SUBJS:
    ni = len(ens_inter[subj])
    nc = len(ens_ictal[subj])
    y  = np.concatenate([np.zeros(ni), np.ones(nc)])
 
    auc_ens   = roc_auc_score(y, np.concatenate([ens_inter[subj], ens_ictal[subj]]))
    auc_recon = roc_auc_score(y[:len(z_recon_inter[subj])+len(z_recon_ictal[subj])],
                               np.concatenate([z_recon_inter[subj], z_recon_ictal[subj]]))
    auc_temp  = roc_auc_score(y,
                               np.concatenate([z_temp_inter[subj][:ni],
                                               z_temp_ictal[subj][:nc]]))
    auc_gamma = roc_auc_score(y,
                               np.concatenate([z_gamma_inter[subj][:ni],
                                               z_gamma_ictal[subj][:nc]]))
    mzi = float(np.median(ens_ictal[subj]))
 
    print(f"{subj:<6} {auc_ens:>10.4f} {mzi:>9.3f} "
          f"{auc_recon:>12.4f} {auc_temp:>11.4f} {auc_gamma:>12.4f}")
 
    per_subject_aurocs.append(auc_ens)
    auroc_rows.append({
        "subject":     subj,
        "auroc_ens":   round(auc_ens,   4),
        "auroc_recon": round(auc_recon, 4),
        "auroc_temp":  round(auc_temp,  4),
        "auroc_gamma": round(auc_gamma, 4),
        "mzi_ens":     round(mzi,       4),
        "n_inter":     ni,
        "n_ictal":     nc,
    })
 
macro_auroc = float(np.mean(per_subject_aurocs))
print(f"\nMACRO AUROC (3-way ensemble): {macro_auroc:.4f}")
print(f"Expected:                     ~0.7957")
print(f"Deviation:                    {macro_auroc - 0.7957:+.4f}")
 
if abs(macro_auroc - 0.7957) > 0.01:
    print("\nWARNING: AUROC deviation > 0.01 — check if signals are properly z-normalized.")
    print("         Common cause: gamma or temporal scores are already z-normalized.")
    print("         If so, skip robust_z_norm for that signal in Section 5A.")
else:
    print("AUROC check PASSED.")
 
# Save verification CSV
df_auroc = pd.DataFrame(auroc_rows)
df_auroc.to_csv(OUT_DIR / "auroc_verification.csv", index=False)
print(f"\nSaved: auroc_verification.csv")

### ⚠ SUPERSEDED — Section 5D (old-weight ens cache export)
Exports `ens_*.npy` consumed by the old `cpd_pipeline_v13`; both superseded (cache quarantined; detection is v14).
Do NOT run; do NOT propagate this cache. Downstream builds the ensemble fresh from components.

In [ ]:
# =============================================================================
# SECTION 5D — EXPORT ENSEMBLE SCORES
# Save ens_inter_{subj}.npy and ens_ictal_{subj}.npy for all 8 test subjects.
# These are the files consumed by cpd_pipeline_v13.py (local, no GPU).
# =============================================================================
print("\n" + "=" * 60)
print("SECTION 5D: Export ensemble scores")
print("=" * 60)
 
for subj in TEST_SUBJS:
    path_inter = OUT_DIR / f"ens_inter_{subj}.npy"
    path_ictal = OUT_DIR / f"ens_ictal_{subj}.npy"
 
    np.save(str(path_inter), ens_inter[subj])
    np.save(str(path_ictal), ens_ictal[subj])
 
    print(f"  Saved: ens_inter_{subj}.npy  ({len(ens_inter[subj])} windows)")
    print(f"  Saved: ens_ictal_{subj}.npy  ({len(ens_ictal[subj])} windows)")
 
print(f"\n{'='*60}")
print("ALL DONE.")
print(f"{'='*60}")
print(f"Total output files: {2 * len(TEST_SUBJS) + 1}")
print(f"  - ens_inter_*.npy : {len(TEST_SUBJS)} files")
print(f"  - ens_ictal_*.npy : {len(TEST_SUBJS)} files")
print(f"  - auroc_verification.csv : 1 file")
print(f"\nNext step: Download all output files from /kaggle/working/")
print(f"           Copy to local results/cpd/scores/ folder")
print(f"           Run: python src/cpd_pipeline_v13.py")
print(f"\nExpected: Macro AUROC = {macro_auroc:.4f} | CPD pen=0.5 → 75% event detection")